# Shared derived fields

A derived field is a named calculation whose function parameters declare its dependencies. `jact` builds one graph from fields declared on the model, the cashflows, and an individual solve. Intensities, payments, and cashflow view weights then read the resolved values through their usual `**kwargs`.

This notebook shows all three scopes, checks the result against equivalent inline formulas, and measures how often a costly feature is constructed during JAX tracing. Run it from a checkout after `pip install -e '.[notebook]'`. The example uses only JAX, NumPy, and `jact`.


In [1]:
import statistics
import time

import jact
import jax
import jax.numpy as jnp
import numpy as np


## 1. Define fields once

The cohort inputs have shape `(batch,)`. The solver passes `t` as a scalar and `d` as a duration grid `(1, D)` or individual point durations `(batch, 1)`. We add `[:, None]` to cohort fields when combining them with `d`.

`attained_age` and `smoker_factor` live on the model because its intensities need them. `risk_surface` uses both, plus `d`; three intensities and one payment will consume it. A field can also depend on a field declared in another scope.


In [2]:
state_space = jact.StateSpace(
    states=["healthy", "disabled", "dead"],
    transitions=[
        ("healthy", "disabled"),
        ("healthy", "dead"),
        ("disabled", "dead"),
    ],
)


def risk_surface(attained_age, smoker_factor, d):
    return (
        0.008
        * jnp.exp(0.035 * (attained_age[:, None] - 50.0))
        * smoker_factor[:, None]
        * (1.0 + 0.08 * d)
    )


model = state_space.build(
    transitions={
        ("healthy", "disabled"): lambda t, d, **kw: 2.0 * kw["risk_surface"],
        ("healthy", "dead"): lambda t, d, **kw: kw["risk_surface"],
        ("disabled", "dead"): lambda t, d, **kw: 1.5 * kw["risk_surface"],
    },
    derived={
        "attained_age": lambda t, entry_age: entry_age + t,
        "smoker_factor": lambda smoker: 1.0 + 0.5 * smoker,
        "risk_surface": risk_surface,
    },
)


The cashflow declaration adds `benefit_index`, which depends on the model's `attained_age`. The solve adds `discount`, which a cashflow view weight uses at the appropriate time. Cashflow fields can be consumed by intensity callables as well; the scope controls ownership, not visibility.


In [3]:
cashflows = state_space.cashflows(
    {
        "premium": jact.cashflows.StateRate(
            {"healthy": lambda t, d, **kw: (
                -250.0 * kw["benefit_index"][:, None] + 0.0 * d
            )}
        ),
        "disability": jact.cashflows.StateRate(
            {"disabled": lambda t, d, **kw: (
                1200.0 * kw["benefit_index"][:, None]
                + 100.0 * kw["risk_surface"]
            )}
        ),
        "death": jact.cashflows.TransitionLump(
            {("healthy", "dead"): lambda t, d, **kw: (
                10000.0 * kw["benefit_index"][:, None] + 0.0 * d
            )}
        ),
    },
    derived={
        "benefit_index": lambda attained_age, base_benefit: (
            base_benefit * (1.0 + 0.01 * (attained_age - 50.0))
        ),
    },
)

views = {
    "raw": jact.cashflows.Raw(),
    "present_value": jact.cashflows.Total(
        weight=lambda t, **kw: kw["discount"], terminal=True
    ),
}

inputs = {
    "entry_age": jnp.array([35.0, 45.0, 55.0, 65.0]),
    "smoker": jnp.array([0.0, 1.0, 0.0, 1.0]),
    "base_benefit": jnp.array([1.0, 1.1, 1.2, 1.3]),
    "interest": 0.03,
}

result = model.solve(
    initial="healthy",
    initial_duration=jnp.array([0.0, 0.25, 0.5, 0.75]),
    horizon=3,
    steps_per_unit=8,
    cashflows=cashflows,
    cashflow_views=views,
    derived={"discount": lambda t, interest: jnp.exp(-interest * t)},
    **inputs,
)

print("probability shape:", result.probability.shape)
print("present values:", np.round(np.asarray(result.cashflows["present_value"]), 2))


probability shape: (25, 4, 3)
present values: [-424.72 -255.46 -329.19  337.95]


## 2. Check the arithmetic

The next model writes the same formulas directly inside each intensity and payment. The equality check makes the performance comparison meaningful: both approaches solve the same problem.


In [4]:
def inline_risk(t, d, kw):
    age = kw["entry_age"] + t
    smoker_factor = 1.0 + 0.5 * kw["smoker"]
    return risk_surface(age, smoker_factor, d)


def inline_benefit(t, kw):
    age = kw["entry_age"] + t
    return kw["base_benefit"] * (1.0 + 0.01 * (age - 50.0))


inline_model = state_space.build(
    transitions={
        ("healthy", "disabled"): lambda t, d, **kw: 2.0 * inline_risk(t, d, kw),
        ("healthy", "dead"): lambda t, d, **kw: inline_risk(t, d, kw),
        ("disabled", "dead"): lambda t, d, **kw: 1.5 * inline_risk(t, d, kw),
    }
)
inline_cashflows = state_space.cashflows(
    {
        "premium": jact.cashflows.StateRate(
            {"healthy": lambda t, d, **kw: (
                -250.0 * inline_benefit(t, kw)[:, None] + 0.0 * d
            )}
        ),
        "disability": jact.cashflows.StateRate(
            {"disabled": lambda t, d, **kw: (
                1200.0 * inline_benefit(t, kw)[:, None]
                + 100.0 * inline_risk(t, d, kw)
            )}
        ),
        "death": jact.cashflows.TransitionLump(
            {("healthy", "dead"): lambda t, d, **kw: (
                10000.0 * inline_benefit(t, kw)[:, None] + 0.0 * d
            )}
        ),
    }
)
inline_views = {
    "raw": jact.cashflows.Raw(),
    "present_value": jact.cashflows.Total(
        weight=lambda t, **kw: jnp.exp(-kw["interest"] * t), terminal=True
    ),
}
inline_result = inline_model.solve(
    initial="healthy",
    initial_duration=jnp.array([0.0, 0.25, 0.5, 0.75]),
    horizon=3,
    steps_per_unit=8,
    cashflows=inline_cashflows,
    cashflow_views=inline_views,
    **inputs,
)

np.testing.assert_allclose(result.probability, inline_result.probability, rtol=2e-5)
jax.tree.map(
    lambda left, right: np.testing.assert_allclose(left, right, rtol=2e-5),
    result.cashflows,
    inline_result.cashflows,
)
print("Derived and inline outputs agree.")


Derived and inline outputs agree.


## 3. Share a costly intermediate

This model averages a nonlinear response over 32 scenarios. Each of four intensities and three payments uses a different scenario weight. In the inline version, every callable runs its own weighted `jax.lax.scan`; those scans have different bodies, so XLA has fewer identical operations to merge. In the derived version, one field returns the common scenario responses and each callable applies its own weights afterward. Both versions calculate the same weighted average.

A Python counter records how often the feature builder runs **while JAX traces the solve**. `jax.make_jaxpr` times tracing separately from compilation and device execution. The cell also measures warm, compiled solves. Timings depend on the compiler, backend, and cohort size.


In [5]:
bench_space = jact.StateSpace(
    ["active", "disabled", "lapsed", "dead"],
    [
        ("active", "disabled"),
        ("active", "lapsed"),
        ("active", "dead"),
        ("disabled", "dead"),
    ],
)
scenario_nodes = jnp.linspace(0.0, 1.0, 32)
trace_calls = {"inline": 0, "shared": 0}


def scenario_base(t, d, entry_age, smoker):
    return (
        0.03 * (entry_age[:, None] + t - 50.0)
        + 0.05 * d
        + 0.1 * smoker[:, None]
    )


def scenario_response(base, node):
    return jax.nn.softplus(base + node * jnp.sin(base * (1.0 + node)))


def shared_responses(t, d, entry_age, smoker):
    trace_calls["shared"] += 1
    base = scenario_base(t, d, entry_age, smoker)
    _, responses = jax.lax.scan(
        lambda _, node: (None, scenario_response(base, node)),
        None,
        scenario_nodes,
    )
    return responses  # (scenario, batch, duration)


def apply_scenario_head(responses, scale):
    weights = 1.0 + 0.01 * scale * scenario_nodes
    average = jnp.sum(weights[:, None, None] * responses, axis=0)
    return scale * (0.01 + 0.001 * average / scenario_nodes.size)


def inline_scenario_value(t, d, kw, scale):
    trace_calls["inline"] += 1
    base = scenario_base(t, d, kw["entry_age"], kw["smoker"])

    def accumulate(total, node):
        weighted = (1.0 + 0.01 * scale * node) * scenario_response(base, node)
        return total + weighted, None

    total, _ = jax.lax.scan(accumulate, jnp.zeros_like(base), scenario_nodes)
    return scale * (0.01 + 0.001 * total / scenario_nodes.size)


def bench_callable(scale, shared):
    if shared:
        return lambda t, d, **kw: apply_scenario_head(
            kw["scenario_responses"], scale
        )
    return lambda t, d, **kw: inline_scenario_value(t, d, kw, scale)


def make_benchmark_case(shared):
    scales = {
        ("active", "disabled"): 1.0,
        ("active", "lapsed"): 0.5,
        ("active", "dead"): 0.8,
        ("disabled", "dead"): 1.2,
    }
    transitions = {
        edge: bench_callable(scale, shared) for edge, scale in scales.items()
    }
    bench_model = bench_space.build(
        transitions=transitions,
        derived={"scenario_responses": shared_responses} if shared else None,
    )
    bench_cashflows = bench_space.cashflows(
        {
            "active_rate": jact.cashflows.StateRate(
                {"active": bench_callable(20.0, shared)}
            ),
            "disabled_rate": jact.cashflows.StateRate(
                {"disabled": bench_callable(50.0, shared)}
            ),
            "death_lump": jact.cashflows.TransitionLump(
                {("active", "dead"): bench_callable(100.0, shared)}
            ),
        }
    )
    return bench_model, bench_cashflows


shared_case = make_benchmark_case(True)
inline_case = make_benchmark_case(False)
bench_inputs = {
    "entry_age": jnp.linspace(35.0, 65.0, 256),
    "smoker": (jnp.arange(256) % 2).astype(jnp.float32),
}


In [6]:
def trace_case(case):
    bench_model, bench_cashflows = case
    start = time.perf_counter()
    jax.make_jaxpr(
        lambda: bench_model.solve(
            initial="active",
            horizon=1,
            steps_per_unit=4,
            cashflows=bench_cashflows,
            **bench_inputs,
        )
    )()
    return time.perf_counter() - start


inline_trace = trace_case(inline_case)
shared_trace = trace_case(shared_case)
print(
    f"feature builder calls while tracing: "
    f"inline={trace_calls['inline']}, shared={trace_calls['shared']}"
)
print(f"tracing time: inline={inline_trace:.3f}s, shared={shared_trace:.3f}s")
print(f"observed tracing ratio (inline/shared): {inline_trace / shared_trace:.2f}x")


def block_result(value):
    for leaf in jax.tree.leaves((value.probability, value.cashflows)):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()


def run_case(case):
    bench_model, bench_cashflows = case
    value = bench_model.solve(
        initial="active",
        horizon=1,
        steps_per_unit=4,
        cashflows=bench_cashflows,
        **bench_inputs,
    )
    block_result(value)
    return value


def time_warm_case(case, repeats=5):
    value = run_case(case)  # Compile before measuring.
    durations = []
    for _ in range(repeats):
        start = time.perf_counter()
        run_case(case)
        durations.append(time.perf_counter() - start)
    return value, statistics.median(durations)


inline_value, inline_warm = time_warm_case(inline_case)
shared_value, shared_warm = time_warm_case(shared_case)
np.testing.assert_allclose(
    shared_value.probability, inline_value.probability, rtol=2e-5
)
jax.tree.map(
    lambda left, right: np.testing.assert_allclose(left, right, rtol=2e-5),
    shared_value.cashflows,
    inline_value.cashflows,
)
print(
    f"median warm call: inline={inline_warm * 1000:.2f}ms, "
    f"shared={shared_warm * 1000:.2f}ms"
)
print(f"observed warm-call ratio (inline/shared): {inline_warm / shared_warm:.2f}x")


feature builder calls while tracing: inline=13, shared=3
tracing time: inline=0.169s, shared=0.124s
observed tracing ratio (inline/shared): 1.36x
median warm call: inline=13.76ms, shared=7.02ms
observed warm-call ratio (inline/shared): 1.96x


### When sharing helps

Derived fields are especially useful when several callables need the same expensive intermediate but apply different final transformations. Here the shared field holds scenario responses; each intensity or payment applies its own scenario weights. The inline scans include those weights, so the compiler cannot simply treat all seven scans as identical. The benchmark measures whether factoring out that common work improves tracing and compiled runtime on your machine.

`jact` resolves input-only fields once per solve or device shard, and time/duration fields once per relevant evaluation context. Keep batch inputs as solve arguments so `jact` can infer and shard the batch size. Field functions should use JAX operations and remain pure; the counter above observes tracing only. Event-time and duration-target callables receive input-only fields, while cashflow view weights receive input-only and time-derived fields.
